# SimRank: Structural Similarity via Co-citation

## Learning Objectives

1. **Define** SimRank and its recursive formulation
2. **Implement** the iterative SimRank algorithm
3. **Explain** the random-walk interpretation of SimRank
4. **Compare** SimRank to PageRank and Jaccard similarity

## Motivation

**Question:** how similar are two nodes in a graph based on their *structural position*?

PageRank measures importance (one node at a time).
Jaccard measures common neighbors (local, symmetric).
**SimRank** measures similarity based on the idea:

> "Two objects are similar if they are referenced by similar objects."

**Applications:**
- Finding similar web pages (co-cited pages)
- Collaborative filtering (users who are rated by similar users)
- Entity matching across databases

## SimRank Definition

$$s(a, a) = 1 \quad \text{(a node is maximally similar to itself)}$$

$$s(a, b) = \frac{C}{|I(a)||I(b)|} \sum_{i=1}^{|I(a)|} \sum_{j=1}^{|I(b)|} s(I_i(a), I_j(b)) \quad \text{for } a \neq b$$

where:
- $I(a)$ = set of **in-neighbors** of $a$ (nodes that link to $a$)
- $C \in (0,1)$ is a decay constant (typically $C = 0.8$)

**Intuition:** pages co-cited by many common pages are similar.
The decay factor $C$ ensures the recursion converges.

In [1]:
from collections import defaultdict

def simrank(adj_in, nodes, C=0.8, n_iter=10):
    """
    Compute SimRank similarity matrix iteratively.
    adj_in: dict node -> list of in-neighbors
    Returns: dict (a,b) -> similarity
    """
    # Initialize: s(a,a)=1, s(a,b)=0 for a≠b
    s = {(a,b): (1.0 if a==b else 0.0) for a in nodes for b in nodes}

    for _ in range(n_iter):
        new_s = {}
        for a in nodes:
            for b in nodes:
                if a == b:
                    new_s[(a,b)] = 1.0
                    continue
                in_a = adj_in.get(a, [])
                in_b = adj_in.get(b, [])
                if not in_a or not in_b:
                    new_s[(a,b)] = 0.0
                    continue
                total = sum(s.get((i,j), 0.0) for i in in_a for j in in_b)
                new_s[(a,b)] = C / (len(in_a) * len(in_b)) * total
        s = new_s
    return s

# Citation graph: papers A,B,C,D cited by X,Y,Z
# X cites A,B; Y cites B,C; Z cites C,D
# So A and B are co-cited by X → similar
# B and C are co-cited by Y → similar
nodes = ['A','B','C','D','X','Y','Z']
adj_in = {
    'A': ['X'],
    'B': ['X','Y'],
    'C': ['Y','Z'],
    'D': ['Z'],
}
s = simrank(adj_in, nodes, C=0.8, n_iter=20)

print("SimRank similarities between cited papers:")
papers = ['A','B','C','D']
print(f"{'':>4}", end="")
for p in papers: print(f"{p:>8}", end="")
print()
for a in papers:
    print(f"{a:>4}", end="")
    for b in papers: print(f"{s[(a,b)]:>8.4f}", end="")
    print()

SimRank similarities between cited papers:
           A       B       C       D
   A  1.0000  0.4000  0.0000  0.0000
   B  0.4000  1.0000  0.2000  0.0000
   C  0.0000  0.2000  1.0000  0.4000
   D  0.0000  0.0000  0.4000  1.0000


## Random Walk Interpretation

**SimRank = expected meeting time of two random walkers.**

Start two independent random walkers at nodes $a$ and $b$ (following in-links backward, i.e., reverse random walk).

$$s(a,b) = \mathbb{E}\left[C^{t_{\text{meet}}}\right]$$

where $t_{\text{meet}}$ is the first time both walkers are at the same node.

- If $a$ and $b$ are the same node: meet immediately → $s = 1$
- If $a$ and $b$ share many common in-neighbors: meet quickly → high $s$
- If $a$ and $b$ are in different parts of the graph: long meeting time → low $s$

In [2]:
# Verify with a Monte Carlo estimate
import random

def simrank_monte_carlo(adj_in, a, b, C=0.8, n_trials=5000, max_steps=50, seed=42):
    """Estimate s(a,b) via random walk simulation."""
    if a == b: return 1.0
    rng = random.Random(seed)
    total = 0.0
    nodes_list = list(adj_in.keys())
    all_nodes = set(nodes_list)
    # Build full in-neighbor dict (with all nodes)
    full_in = {n: adj_in.get(n,[]) for n in nodes_list}

    for _ in range(n_trials):
        wa, wb = a, b
        for t in range(1, max_steps+1):
            # Step both walkers backward
            in_a = full_in.get(wa,[])
            in_b = full_in.get(wb,[])
            if not in_a or not in_b: break
            wa = rng.choice(in_a)
            wb = rng.choice(in_b)
            if wa == wb:
                total += C**t
                break
    return total / n_trials

# Compare iterative vs Monte Carlo
print("Iterative vs Monte Carlo SimRank:")
for pair in [('A','B'),('A','C'),('A','D'),('B','C')]:
    a, b = pair
    it  = s[(a,b)]
    mc  = simrank_monte_carlo(adj_in, a, b, C=0.8, n_trials=10000)
    print(f"  s({a},{b}): iterative={it:.4f}  MC={mc:.4f}")

Iterative vs Monte Carlo SimRank:
  s(A,B): iterative=0.4000  MC=0.3997
  s(A,C): iterative=0.0000  MC=0.0000
  s(A,D): iterative=0.0000  MC=0.0000
  s(B,C): iterative=0.2000  MC=0.1961


## Comparison with Other Similarity Measures

| Measure | Computes | Symmetric? | Range | Uses |
|---------|---------|-----------|-------|------|
| Jaccard | Common neighbors / union | Yes | [0,1] | Local structure |
| Adamic-Adar | Weighted common neighbors | Yes | [0,∞) | Social networks |
| PageRank | Importance (not similarity) | No | [0,1] | Ranking |
| **SimRank** | Structural equivalence via in-links | Yes | [0,1] | Co-citation |

SimRank is **symmetric**: $s(a,b) = s(b,a)$ — follows from the definition.
PageRank is **asymmetric**: PR(a) ≠ PR(b) in general.

**Computational cost:** O(n²) per iteration — expensive for large graphs.
**Approximations:** sample random walk pairs; use fingerprint-based SimRank (SimFP).